# 15.4 Fixtures, Isolation and Test Data

**Prerequisites:** 15.3 pytest, 6.3 Context Managers, 08 File Handling  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- What a fixture is: **dependency injection**, requested by name
- `yield` fixtures — setup and teardown in one function (**6.3** again)
- The four scopes, and the exact order they set up and tear down in
- 🔴 The session-scoped mutable fixture — the classic shared-state bug
- Factory fixtures, parametrised fixtures, and `autouse`
- The built-ins you get free: `tmp_path`, `capsys`, `monkeypatch`, `caplog`, `recwarn`
- `conftest.py` — sharing fixtures, and what `rootdir` means
- 🔴 Why fixtures do **not** work on `unittest.TestCase` subclasses

---

## The problem

`setUp` (**15.2**) gives every test in a class the same preparation. That is useful and also
too blunt:

- **All or nothing.** Every test pays for every piece of setup, even the ones that need none.
- **No sharing across classes.** Two `TestCase` classes needing the same database mean
  copy-paste, or an inheritance hierarchy you will regret.
- **No layering.** A cheap thing (a temp file) and an expensive thing (a database) have
  different natural lifetimes, but `setUp` runs at exactly one rhythm.
- **Invisible dependencies.** Reading a test tells you nothing about what `setUp` prepared.

Fixtures fix all four. A test **declares what it needs as a parameter**, and `pytest` supplies
it:

```
@pytest.fixture                       ┌─ the fixture's name...
def spool_dir(tmp_path):              │
    path = tmp_path / "spool"         │
    path.mkdir()                      │
    return path                       │
                                      ▼
def test_writes_a_job_file(spool_dir):     ...is the parameter name
    (spool_dir / "job.json").write_text("{}")
    assert len(list(spool_dir.iterdir())) == 1
```

No import, no base class, no registration. The **name is the wiring**. That is dependency
injection, and it means the test signature is an honest list of its requirements.

In [ ]:
import shutil
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="py154_"))


def make_project(files, name="proj"):
    """Create a temp project from {relative path: source} and return its path."""
    project = Path(tempfile.mkdtemp(prefix=f"{name}_", dir=WORK))
    for relpath, source in files.items():
        path = project / relpath
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    return project


def pytest_in(project, *args, keep_cache=False):
    """Run pytest inside `project` and return its output, with the command echoed."""
    cmd = [sys.executable, "-m", "pytest", "--no-header"]
    if not keep_cache:
        cmd += ["-p", "no:cacheprovider"]
    cmd += list(args)
    done = subprocess.run(cmd, cwd=project, capture_output=True, text=True,
                          encoding="utf-8", errors="replace", timeout=300)
    return (f"$ pytest {' '.join(args)}".rstrip() + "\n" + "-" * 70 + "\n"
            + (done.stdout + done.stderr).rstrip()
            + "\n" + "-" * 70 + f"\nexit code: {done.returncode}")


print("scratch:", WORK)

## `yield` — setup, hand over, tear down

A fixture that `yield`s instead of `return`ing gets a teardown half. Everything before the
`yield` is setup; everything after runs when the fixture's scope ends — **even if the test
fails**.

```
@pytest.fixture
def connection():
    conn = connect()        ← setup
    yield conn              ← the test runs here, with `conn` as its argument
    conn.close()            ← teardown, guaranteed
```

If that shape looks familiar, it is: this is exactly a `@contextmanager` from **6.3**. A
fixture *is* a context manager whose lifetime `pytest` manages for you.

🔴 The guarantee has one hole worth knowing: if the setup half **raises**, the teardown half
never runs, because execution never reached the `yield`. Anything you allocate before a risky
step should be released with `request.addfinalizer` or its own nested fixture.

In [ ]:
yield_demo = make_project({
    "test_yield.py": r"""
        import pytest

        EVENTS = []


        @pytest.fixture
        def connection():
            EVENTS.append("  setup: open connection")
            yield {"open": True}
            EVENTS.append("  teardown: close connection")


        def test_uses_the_connection(connection):
            EVENTS.append("    test A runs")
            assert connection["open"]


        def test_that_fails_still_gets_teardown(connection):
            EVENTS.append("    test B runs (about to fail)")
            assert connection["open"] is False


        @pytest.fixture(scope="session", autouse=True)
        def _print_events():
            yield
            print("\n--- what actually happened ---")
            for event in EVENTS:
                print(event)
    """,
}, name="yield")

print(pytest_in(yield_demo, "-q", "-s", "--tb=line"))

Read the trace: **test B failed and its teardown still ran.** That is the
whole reason to prefer a fixture over writing setup and cleanup by hand in the test body.

## Scopes

A fixture's `scope` decides how often it is built.

| Scope | Built | Use for |
|---|---|---|
| `function` (default) | once per test | anything cheap and mutable |
| `class` | once per test class | shared, read-only setup for a group |
| `module` | once per test file | a schema, a loaded data file |
| `package` | once per package | rarely needed |
| `session` | once per `pytest` run | a database engine, a spun-up server |

The rule of thumb: **the widest scope you can justify, and never wider.** Every step wider buys
speed and costs isolation.

The next cell chains three fixtures — `connection` (function) depends on `schema` (module)
depends on `db_engine` (session) — and prints the order as it happens.

In [ ]:
scopes = make_project({
    "conftest.py": r"""
        import pytest

        ORDER = []


        @pytest.fixture(scope="session")
        def db_engine():
            ORDER.append("session   setup    db_engine")
            yield "engine"
            ORDER.append("session   teardown db_engine")


        @pytest.fixture(scope="module")
        def schema(db_engine):
            ORDER.append("  module  setup    schema")
            yield "schema"
            ORDER.append("  module  teardown schema")


        @pytest.fixture
        def connection(schema):
            ORDER.append("    func  setup    connection")
            yield "conn"
            ORDER.append("    func  teardown connection")


        @pytest.fixture(scope="session", autouse=True)
        def _report():
            yield
            print("\n--- setup / teardown order ---")
            for line in ORDER:
                print(line)
    """,
    "test_scopes.py": r"""
        def test_one(connection):
            assert connection == "conn"


        def test_two(connection):
            assert connection == "conn"
    """,
}, name="scopes")

print(pytest_in(scopes, "-q", "-s"))

Session sets up first and tears down last; the function-scoped `connection`
was built and destroyed **twice**, once per test, while `db_engine` was built **once**. Nested
lifetimes, exactly like nested `with` blocks.

`--setup-show` gives you the same picture for any real suite, without instrumenting anything:

In [ ]:
print(pytest_in(scopes, "--setup-show", "-q"))

`S`, `M` and `F` are the scopes. This is the first thing to run when a test
is slow — it shows you what is being rebuilt per test that need not be.

## 🔴 The session-scoped mutable fixture

This is the shared-state bug from **15.1**, wearing a fixture as a disguise. It is the single
most common way a pytest suite becomes order-dependent.

In [ ]:
trap = make_project({
    "test_shared.py": r"""
        import pytest


        @pytest.fixture(scope="session")
        def registry():
            return {"jobs": []}          # 🔴 built once; every test shares this dict


        def test_a_adds_a_job(registry):
            registry["jobs"].append("build-1")
            assert len(registry["jobs"]) == 1


        def test_b_expects_it_empty(registry):
            assert registry["jobs"] == []
    """,
}, name="trap")

print(pytest_in(trap, "-q"))

`test_b` fails, and it would **pass** if it ran first. Note how the failure
report prints `registry = {'jobs': ['build-1']}` — pytest shows you the fixture's value, which
is usually enough to spot the problem.

### Three ways out, in order of preference

| Fix | When |
|---|---|
| **Make it function-scoped** | the default answer; do this unless it is measurably too slow |
| **Return a factory** | the setup is expensive but each test needs its own *instance* |
| **Reset in the fixture's teardown** | you genuinely must share one object (a real DB connection) |

A **factory fixture** returns a *function*. The expensive part happens once; the cheap
per-test object is made fresh on demand. Note that the factory can also collect what it
creates, so its teardown cleans up everything the test made.

In [ ]:
factory = make_project({
    "test_factory.py": r"""
        import pytest


        @pytest.fixture(scope="session")
        def id_prefix():
            return "build"                      # expensive-to-compute, genuinely read-only


        @pytest.fixture
        def make_job(id_prefix):
            created = []

            def _make(number, state="queued"):
                job = {"id": f"{id_prefix}-{number}", "state": state, "attempts": 0}
                created.append(job)
                return job

            yield _make
            print(f"      teardown: cleaning up {len(created)} job(s)")


        def test_a_creates_two_jobs(make_job):
            first, second = make_job(1), make_job(2)
            assert first["id"] == "build-1"
            assert second["state"] == "queued"


        def test_b_gets_fresh_ones(make_job):
            job = make_job(1)
            assert job["attempts"] == 0         # no leakage from test_a
    """,
}, name="factory")

print(pytest_in(factory, "-q", "-s"))

## The built-in fixtures

`pytest` ships fixtures you never have to write. These five cover most of what tests need from
the outside world:

| Fixture | Gives you | Replaces |
|---|---|---|
| `tmp_path` | a fresh `pathlib.Path` directory, per test | `tempfile` boilerplate (**08**) |
| `tmp_path_factory` | the same, session-scoped | |
| `capsys` | captured `stdout`/`stderr` | redirecting streams by hand |
| `monkeypatch` | attribute/env/`sys.path` patching, **auto-undone** | `try/finally` restore dances |
| `caplog` | captured log records | `assertLogs` (**15.2**) |
| `recwarn` | captured warnings | `assertWarns` (**15.2**) |
| `request` | metadata about the running test | |

🔴 **`monkeypatch` undoes itself at the end of the test.** That is the entire reason to prefer
it over `os.environ[...] = ...`, which leaks into every test that runs after. The cell below
proves the restoration with a second test.

In [ ]:
builtins_demo = make_project({
    "test_builtins.py": r"""
        import logging
        import os
        import warnings

        import pytest


        def test_tmp_path_is_a_real_fresh_directory(tmp_path):
            spool = tmp_path / "spool"
            spool.mkdir()
            (spool / "build-1.json").write_text('{"state": "queued"}', encoding="utf-8")
            print("      tmp_path:", tmp_path.name)
            assert len(list(spool.iterdir())) == 1


        def test_tmp_path_is_different_next_time(tmp_path):
            print("      tmp_path:", tmp_path.name)
            assert list(tmp_path.iterdir()) == []


        def test_capsys_captures_printed_output(capsys):
            print("processing job build-42")
            captured = capsys.readouterr()
            assert captured.out == "processing job build-42\n"
            assert captured.err == ""


        def test_monkeypatch_sets_an_env_var(monkeypatch):
            monkeypatch.setenv("RETRY_CEILING", "5")
            assert os.environ["RETRY_CEILING"] == "5"


        def test_the_env_var_is_gone_again():
            assert "RETRY_CEILING" not in os.environ      # 🔴 auto-undone


        def test_caplog_captures_log_records(caplog):
            with caplog.at_level(logging.WARNING, logger="worker"):
                logging.getLogger("worker").warning("budget exhausted for %s", "build-42")
            assert "budget exhausted" in caplog.text
            assert caplog.records[0].levelname == "WARNING"


        def test_recwarn_captures_warnings(recwarn):
            warnings.warn("the `retries` argument is deprecated", DeprecationWarning)
            assert len(recwarn) == 1
            assert recwarn[0].category is DeprecationWarning
    """,
}, name="builtins")

print(pytest_in(builtins_demo, "-q", "-s"))

Two different `tmp_path` names in that output — a fresh directory per test,
created under the system temp dir, and `pytest` keeps the last few runs around so you can
inspect them after a failure.

> This is what **08 File Handling**'s "never write into the repository" rule looks like when a
> framework does it for you. Every notebook in these notes does the same thing by hand with
> `tempfile.mkdtemp`.

## `conftest.py` — sharing fixtures

A fixture defined in a test file is visible only in that file. Move it to **`conftest.py`** and
every test file in that directory *and below* can request it. No import needed — `pytest`
loads `conftest.py` automatically.

```
tests/
├── conftest.py            fixtures for every test below
├── unit/
│   ├── conftest.py        extra fixtures, unit tests only
│   └── test_retry.py
└── integration/
    └── test_database.py   sees tests/conftest.py, not unit/conftest.py
```

🔴 **`conftest.py` is also what defines the `rootdir`** — along with `pytest.ini`,
`pyproject.toml`, `tox.ini` or `setup.cfg`. `rootdir` is how `pytest` resolves relative paths
and works out how to import your test modules. Nearly every "works locally, fails in CI"
import error is really a `rootdir` disagreement. See **07** and **18**.

In [ ]:
shared = make_project({
    "conftest.py": r"""
        import pytest


        @pytest.fixture
        def job():
            return {"id": "build-1", "state": "queued", "attempts": 0}


        @pytest.fixture
        def spool(tmp_path):
            path = tmp_path / "spool"
            path.mkdir()
            return path
    """,
    "test_state.py": r"""
        def test_a_new_job_is_queued(job):
            assert job["state"] == "queued"
    """,
    "test_spooling.py": r"""
        import json


        def test_job_can_be_spooled(job, spool):
            (spool / f"{job['id']}.json").write_text(json.dumps(job), encoding="utf-8")
            written = json.loads((spool / "build-1.json").read_text(encoding="utf-8"))
            assert written["state"] == "queued"
    """,
}, name="shared")

print(pytest_in(shared, "-v"))
print()
print("Two files, no imports between them, both using fixtures from conftest.py.")

### Finding out what fixtures exist

```bash
pytest --fixtures            # every available fixture, with its docstring
pytest --fixtures-per-test   # which fixtures each test actually uses
```

Worth running on an unfamiliar codebase — it is the fastest map of what the test suite can
give you.

In [ ]:
print(pytest_in(shared, "--fixtures-per-test", "-q"))

## Parametrised fixtures and `autouse`

**`params=`** on a fixture runs *every test that uses it* once per parameter. Where
`@parametrize` (**15.3**) varies the data a single test sees, a parametrised fixture varies
the **environment** a whole group of tests runs in — the classic use being "run the entire
suite against SQLite and against PostgreSQL".

**`autouse=True`** applies a fixture without any test asking for it. Convenient, and easy to
abuse: it makes setup invisible again, which is the very problem fixtures solved. Reserve it
for genuinely cross-cutting concerns — resetting a global, freezing a clock, clearing a cache.

In [ ]:
params_demo = make_project({
    "test_params.py": r"""
        import pytest

        CALLS = []


        @pytest.fixture(params=["sqlite", "postgres"])
        def backend(request):
            CALLS.append(f"setup {request.param}")
            return request.param


        @pytest.fixture(autouse=True)
        def _reset_between_tests():
            # runs for every test in this file, requested or not
            yield


        def test_backend_can_store(backend):
            assert backend in {"sqlite", "postgres"}


        def test_backend_can_query(backend):
            assert isinstance(backend, str)


        def test_does_not_use_the_backend():
            assert True


        @pytest.fixture(scope="session", autouse=True)
        def _report():
            yield
            print("\n  fixture was set up", len(CALLS), "times:", CALLS)
    """,
}, name="params")

print(pytest_in(params_demo, "-v", "-s"))

Two tests × two params = **four** generated tests, with the parameter in the
node ID (`test_backend_can_store[sqlite]`). The third test did not request `backend`, so it ran
**once** — the fixture only multiplies the tests that actually need it.

## 🔴 Fixtures and `unittest.TestCase` do not mix

`pytest` runs `unittest.TestCase` tests (**15.2**, **15.3**) — but those tests **cannot request
fixtures as arguments**. The `unittest` runner controls how test methods are called, and it
does not know how to inject anything.

This is the one genuine limit of "adopt pytest, keep your existing tests". It is worth seeing
the actual error, because the message is not obvious.

In [ ]:
mixing = make_project({
    "conftest.py": r"""
        import pytest


        @pytest.fixture
        def spool(tmp_path):
            path = tmp_path / "spool"
            path.mkdir()
            return path
    """,
    "test_mixing.py": r"""
        import unittest


        def test_a_plain_function_gets_the_fixture(spool):
            assert spool.exists()


        class SpoolTests(unittest.TestCase):
            def test_a_testcase_method_does_not(self, spool):
                self.assertTrue(spool.exists())
    """,
}, name="mixing")

print(pytest_in(mixing, "-q", "--tb=short"))

The plain function got its fixture; the `TestCase` method failed. 🔴 Read the
error carefully — it does **not** say "fixtures are unsupported here":

```
TypeError: SpoolTests.test_a_testcase_method_does_not() missing 1 required
           positional argument: 'spool'
```

Nothing injected anything, so `unittest` simply called `self.test_...()` with no arguments and
Python complained about the signature. The traceback points into `unittest/case.py`, not into
pytest — which is exactly why this costs people an afternoon the first time.

### Your options

| If you want | Do this |
|---|---|
| fixtures on those tests | convert the class to plain functions |
| to keep `TestCase` | use `setUp`, or an `autouse` fixture that sets `self.something` |
| `tmp_path` specifically | `unittest` has `enterContext(TemporaryDirectory())` since 3.11 |

## When *not* to use a fixture

Fixtures are not free — they add indirection, and a test whose meaning lives in three
`conftest.py` files is hard to read.

- If the setup is **one line** and used by **one test**, write it in the test.
- If the setup is really **the thing being tested**, it belongs in the test body — a fixture
  would hide the Arrange step you are trying to demonstrate.
- If you find yourself passing flags into a fixture to make it behave differently, you want a
  **factory** fixture instead.

In [ ]:
# ---- tidy up ----
shutil.rmtree(WORK, ignore_errors=True)
print("scratch removed:", not WORK.exists())

---

## Common Mistakes & Pitfalls

1. 🔴 **A mutable object from a `session`- or `module`-scoped fixture.** Every test shares it, so the suite becomes order-dependent — the exact bug from **15.1**. Use function scope, or a factory.
2. 🔴 **Expecting fixtures to work on `unittest.TestCase` methods.** They do not, and the error blames the argument count rather than naming the real cause.
3. **Allocating something before a risky step in a `yield` fixture.** If setup raises before the `yield`, the teardown half never runs. Use `addfinalizer` or a nested fixture.
4. **Overusing `autouse`.** It reintroduces invisible setup, which is what fixtures were supposed to eliminate.
5. **Setting `os.environ` directly instead of `monkeypatch.setenv`.** The change leaks into every test that runs afterwards.
6. **Reaching for the widest scope for speed.** `session` scope on something mutable buys milliseconds and costs you a day of debugging.
7. **Deep `conftest.py` hierarchies.** A fixture defined three directories up, overridden twice, is worse than a little duplication.
8. **A fixture that hides the Arrange step being demonstrated.** If setup *is* the subject of the test, leave it in the test.
9. **Writing into the repository instead of `tmp_path`.** Tests that leave files behind pass once and fail on the rerun.

## Best Practices

- Default to `function` scope; widen only when `--setup-show` or `--durations` proves it matters.
- Prefer a **factory fixture** over a wide-scoped mutable one — expensive setup shared, cheap instances fresh.
- Use `yield` for anything needing cleanup; it is a context manager (**6.3**) that pytest manages.
- Put shared fixtures in `conftest.py`, as close to the tests that use them as possible.
- Use `tmp_path` rather than `tempfile` by hand, and `monkeypatch` rather than manual save/restore.
- Give fixtures docstrings — they show up in `pytest --fixtures`.
- Run `--setup-show` when a suite feels slow, and `--fixtures-per-test` when it feels mysterious.
- Name fixtures after the thing they provide (`spool`, `db_engine`), not after what they do (`setup_spool`).

## Practice Exercises

Try these before moving on.

1. Write a `spool` fixture on `tmp_path` and two tests that each write a file, then assert in both that the directory contained nothing at the start. Prove they pass in either order.
2. 🔴 Take the `registry` trap from this notebook and fix it three ways: function scope, a factory, and teardown that resets. Which reads best, and which would you use on a fixture that takes 2 seconds to build?
3. Chain three fixtures with `session`, `module` and `function` scope and predict the setup/teardown order before running `--setup-show`. Were you right?
4. Convert `15.2`'s `CleanupTests` (which uses `setUp` and `addCleanup`) into pytest fixtures. What happens to `addCleanup`?
5. Write a parametrised fixture with `params=["json", "csv"]` and one test that works for both formats. Then add a test that does not use it and confirm it runs only once.
6. Use `monkeypatch.setenv` to set a config variable, and write a second test proving it is gone. Now do the same thing with `os.environ` directly and watch the second test fail.
7. 🔴 Write a `yield` fixture whose setup raises **after** allocating a temp directory. Confirm the directory is left behind, then fix it with `request.addfinalizer`.
8. **Interview question:** what is the difference between a fixture with `scope="session"` and a module-level global? When is the fixture genuinely better?

---

## Version notes

| Version | Change |
|---|---|
| **pytest 8** | `tmpdir` (the old `py.path.local` fixture) fully superseded by `tmp_path`; use `tmp_path` in new code |
| **pytest 7** | `pytest.approx` dict support; `tmp_path_factory` stabilised |
| **pytest 6** | `--strict-markers` and `-p no:cacheprovider` behaviour as used throughout this folder |
| **Python 3.11** | `unittest.TestCase.enterContext()` — the closest `unittest` equivalent to a `yield` fixture |
| **Python 3.9** | `unittest.addClassCleanup()` — the class-scope partner of `addCleanup` |

## Where next

| Notebook | Covers |
|---|---|
| **15.5 Test Doubles** | replacing the network, the clock and the database — with `monkeypatch` and `unittest.mock` |
| **15.6 Testing in Practice** | coverage, property-based testing, layout, CI |

## Related

- **15.1** — the order-dependence bug that session-scoped fixtures recreate
- **15.2** — `setUp`/`tearDown`/`addCleanup`, which fixtures replace
- **15.3** — `@parametrize`, the data-level partner of parametrised fixtures
- **6.3 Context Managers** — what a `yield` fixture actually is
- **08 File Handling** — the `tempfile` work `tmp_path` does for you
- **07 Module and Packages** / **18 Tooling** — imports, `sys.path`, and `rootdir`